# 📈 01: Análisis de Mercado, Precios y Liquidez
Este notebook analiza la microestructura de mercado de Brokerdex mediante **consultas SQL puras**.

**Preguntas clave a responder:**
1. ¿Cuáles son los activos más caros y cuáles tienen mayor desviación respecto a su precio de referencia?
2. ¿Cómo se comportan las velas japonesas (OHLCV) y el volumen en diferentes intervalos (1H / 1D)?
3. ¿Qué criaturas concentran el mayor volumen negociado en dólares?


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Conexión pura a PostgreSQL
from db import get_engine
engine = get_engine()


## 1. Catálogo de Criaturas y Dispersión de Precios
Consultamos el estado actual de todas las criaturas registradas.


In [ ]:
query_catalog = """
SELECT 
    name,
    type,
    current_price,
    pricing_reference_price,
    ROUND(((current_price - pricing_reference_price) / NULLIF(pricing_reference_price, 0)) * 100, 2) AS premium_pct,
    wins,
    losses
FROM creature_creature
ORDER BY current_price DESC;
"""

df_catalog = pd.read_sql_query(query_catalog, engine)
df_catalog


## 2. Gráfico Interactivo de Velas (Candlesticks) con Volumen
Seleccionamos una criatura para visualizar su historial de velas horarias (1H) con subgráfico de volumen.


In [ ]:
target_creature = 'Charizard'

query_ohlcv = f"""
SELECT 
    timestamp,
    open_price,
    high_price,
    low_price,
    close_price,
    volume
FROM trading_pricehistory ph
JOIN creature_creature c ON ph.creature_id = c.id
WHERE c.name = '{target_creature}' AND ph.interval = '1H'
ORDER BY timestamp ASC;
"""

df_candles = pd.read_sql_query(query_ohlcv, engine)
df_candles['timestamp'] = pd.to_datetime(df_candles['timestamp'])

# Crear gráfico interactivo con subgráficos de velas y volumen
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.05, row_heights=[0.75, 0.25],
    subplot_titles=(f"Velas OHLCV 1H - {target_creature}", "Volumen de Unidades Negociadas")
)

# Velas japonesas
fig.add_trace(
    go.Candlestick(
        x=df_candles['timestamp'],
        open=df_candles['open_price'],
        high=df_candles['high_price'],
        low=df_candles['low_price'],
        close=df_candles['close_price'],
        name="OHLC"
    ),
    row=1, col=1
)

# Volumen
fig.add_trace(
    go.Bar(
        x=df_candles['timestamp'],
        y=df_candles['volume'],
        name="Volumen",
        marker_color="rgba(100, 150, 250, 0.6)"
    ),
    row=2, col=1
)

fig.update_layout(xaxis_rangeslider_visible=False, height=650, template="plotly_dark")
fig.show()


## 3. Top de Criaturas por Volumen Negociado en Dólares ($)
Consultamos los trades ejecutados agrupados por criatura.


In [ ]:
query_volume = """
SELECT 
    c.name,
    c.type,
    COUNT(t.id) AS total_trades,
    ROUND(SUM(t.quantity), 4) AS units_traded,
    ROUND(SUM(t.quantity * t.price_per_unit), 2) AS total_volume_usd,
    ROUND(AVG(t.price_per_unit), 2) AS avg_trade_price
FROM trading_trade t
JOIN creature_creature c ON t.creature_id = c.id
GROUP BY c.id, c.name, c.type
ORDER BY total_volume_usd DESC;
"""

df_volume = pd.read_sql_query(query_volume, engine)

fig_vol = px.bar(
    df_volume,
    x='total_volume_usd',
    y='name',
    orientation='h',
    color='type',
    text='total_volume_usd',
    title="Ranking de Volumen Total Negociado (USD)",
    labels={'total_volume_usd': 'Volumen Total ($)', 'name': 'Criatura'},
    template="plotly_dark"
)
fig_vol.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig_vol.show()
